# Pipeline Inspection: Tuning Pipeline (Optuna Two-Phase Hyperparameter Search)

This notebook executes the `TuningPipeline` (`pipelines/tuning_pipeline/pipeline.py`) and its components stage by stage, and verifies that each step produces the contracted output before the next stage consumes it.

The tuning pipeline searches for the best hyperparameters of a per-pixel Gaussian-mixture TomoSAR reconstruction model using **Optuna**. It runs in **two sequential phases** driven by the orchestrator in `main/tune.py`:

- **Phase 1 — learning dynamics.** Samples the per-parameter-group learning rates, weight decays and dropout (`model_config_cls.tunable_lr_params()`) and trains a short model per trial, minimising the best validation loss.
- **Phase 2 — architecture.** Freezes the best learning-dynamics hyperparameters found in Phase 1 and searches the architectural search space (`model_config_cls.tunable_arch_params()` — feature widths, bottleneck factor, activation, normalization, upsampling), again minimising the best validation loss.

Each phase is an Optuna `study` with a `TPESampler` (multivariate, constant-liar, seeded) and a `MedianPruner`. Each trial trains a `TrialPipeline` (a `TrainingPipeline` subclass) for `n_epochs` epochs and reports the validation loss back to Optuna after every epoch through `TrialTrainer._trial_callback`, so unpromising trials are pruned early.

**What this notebook verifies, stage by stage:**

- The Optuna study is created with the correct direction (`minimize`), sampler, pruner and persistent SQLite storage, and is shared across the four GPU workers per phase.
- The `ParamSampler` materialises exactly the search space declared by the model config, with float ranges and categorical choices matching the config.
- The per-trial config assembly (`BaseTuner._objective`) deep-copies the base trainer / dataset configs and overrides epochs, scheduler epochs, early-stop patience and the per-trial log directory, so trials never mutate shared state.
- A single trial trains a `TrialPipeline`, reports validation loss every epoch, and is pruned by the `MedianPruner` when it falls behind the running median.
- The Phase-1 loop populates the study with completed and pruned trials and exposes a `best_trial`.
- The Phase-1 best parameters are decoded (resolving `indexed_categorical` `__idx` keys) and fed as a frozen seed into Phase 2.
- The Phase-2 loop searches only the architectural space, with the Phase-1 best learning dynamics held fixed, and the two phases' search spaces are disjoint.

> This notebook does **not** require a GPU run to be present. Where a real training loop would be invoked (single trial, full phase loops) we use a lightweight closed-form surrogate objective with the **identical search space, sampler and pruner**, so the Optuna mechanics, search-space coverage, pruning behaviour and best-trial extraction are all exercised and asserted faithfully. Every surrogate is clearly labelled and every assertion is derived from the config, never hardcoded.


In [ ]:
from __future__ import annotations

import sys
import copy
from pathlib import Path

import numpy as np
import optuna
import matplotlib as mpl
import matplotlib.pyplot as plt

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from configuration.tuning_config            import TuningConfig, Phase1TuneConfig, Phase2TuneConfig
from configuration.models_config            import UNetConfig
from models                                  import CONFIG_REGISTRY
from pipelines.tuning_pipeline.pipeline      import TuningPipeline
from pipelines.tuning_pipeline.base_tuner    import BaseTuner
from pipelines.tuning_pipeline.param_sampler import ParamSampler
from pipelines.tuning_pipeline.phase1_tuner  import Phase1Tuner
from pipelines.tuning_pipeline.phase2_tuner  import Phase2Tuner
from pipelines.tuning_pipeline.trial_pipeline import TrialPipeline
from pipelines.tuning_pipeline.trial_trainer import TrialTrainer

mpl.rcParams.update({
    "font.family"        : "serif",
    "font.size"          : 11,
    "axes.labelsize"     : 12,
    "axes.titlesize"     : 13,
    "legend.fontsize"    : 10,
    "xtick.labelsize"    : 10,
    "ytick.labelsize"    : 10,
    "figure.dpi"         : 150,
    "savefig.dpi"        : 300,
    "savefig.bbox"       : "tight",
    "axes.spines.top"    : False,
    "axes.spines.right"  : False,
})

optuna.logging.set_verbosity(optuna.logging.WARNING)

figure_directory = project_root / "notebooks" / "figures" / "inspect_tuning_pipeline"
figure_directory.mkdir(parents=True, exist_ok=True)

palette_phase1 = "#2a6f97"
palette_phase2 = "#bc6c25"
palette_pruned = "#9d0208"
palette_neutral = "#495057"

tuning_configuration = TuningConfig()
model_name_under_inspection = "unet"
model_config_class = CONFIG_REGISTRY[model_name_under_inspection]

print("model under inspection :", model_name_under_inspection)
print("model config class     :", model_config_class.__name__)
print("figure directory       :", figure_directory)
print()
print("Phase-1 trials          :", tuning_configuration.phase1.n_trials)
print("Phase-1 epochs / trial  :", tuning_configuration.phase1.n_epochs)
print("Phase-1 early-stop pat. :", tuning_configuration.phase1.early_stop_patience)
print("Phase-2 trials          :", tuning_configuration.phase2.n_trials)
print("Phase-2 epochs / trial  :", tuning_configuration.phase2.n_epochs)
print("GPUs (workers / phase)  :", tuning_configuration.n_gpus)


## Pipeline Overview

`TuningPipeline` exposes two orchestrated entry points, `run_phase1(study, n_trials)` and `run_phase2(study, n_trials, best_phase1_params)`. The scheduler in `main/tune.py` creates a fresh Optuna study per phase, spawns one worker per GPU that loads the shared study and calls the matching `run_phase*`, then extracts the best trial of Phase 1 to seed Phase 2. Each arrow below is a handoff of a concrete object from one component to the next.

```
TuningConfig + model_config_cls
  └─► Stage 1  optuna.create_study()           →  Study  (direction=minimize, TPESampler, MedianPruner, SQLite)
        └─► Stage 2  ParamSampler.sample()       →  dict of sampled hyperparameters (Phase-1 search space)
              └─► Stage 3  BaseTuner._objective() →  per-trial deep-copied trainer/dataset cfg + TrialPipeline
                    └─► Stage 4  TrialTrainer._trial_callback() →  per-epoch report + prune decision
                          └─► Stage 5  Phase1Tuner.run() / study.optimize() →  Study populated with trials
                                └─► Stage 6  study.best_trial + _decode_phase1_best() →  frozen Phase-1 best params
                                      └─► Stage 7  Phase2Tuner.run() →  Phase-2 study over architecture space
                                            └─► best_phase2_trial  →  merged best configuration
```

| Stage | Callable | Input | Output |
|-------|----------|-------|--------|
| 1 | `optuna.create_study()` (driven by `main/tune.py::_create_study`) | `TuningConfig.phase*`, storage URL | `optuna.Study` (minimize, TPESampler, MedianPruner) |
| 2 | `ParamSampler.sample()` | `optuna.Trial`, `tunable_lr_params()` | `dict[str, float]` sampled Phase-1 hyperparameters |
| 3 | `BaseTuner._objective()` (config assembly) | `optuna.Trial` | deep-copied trainer/dataset config + `TrialPipeline` |
| 4 | `TrialTrainer._trial_callback()` | `val_loss`, `epoch` | `trial.report()` + optional `TrialPruned` |
| 5 | `Phase1Tuner.run()` / `study.optimize()` | `Study`, `n_trials` | `Study` populated with complete + pruned trials |
| 6 | `study.best_trial` + `_decode_phase1_best()` | Phase-1 `Study` | frozen `dict` of best Phase-1 params |
| 7 | `Phase2Tuner.run()` / `study.optimize()` | `Study`, `n_trials`, Phase-1 best | `Study` over `tunable_arch_params()` with Phase-1 best fixed |

The two phases are deliberately treated as **distinct stages** (Stage 5 and Stage 7) because they use different search spaces, different tuner subclasses and a one-directional handoff (Phase-1 best params -> Phase-2 seed).


---
## Stage 1: Study creation — sampler, pruner, direction, storage

**Callable:** `optuna.create_study()` (as configured by `main/tune.py::TuningOrchestrator._create_study()`)
**Input:** a phase config (`TuningConfig.phase1` / `.phase2`) supplying `pruner_n_startup_trials` and `pruner_n_warmup_steps`, plus the persistent storage URL.
**Output:** an `optuna.Study` configured to **minimize** the objective, with a multivariate constant-liar `TPESampler` (seed 42) and a `MedianPruner`.

The study is the shared optimisation state. Optuna's Tree-structured Parzen Estimator models the objective as two densities over the search space — $l(\mathbf{x})$ for configurations whose objective is below a quantile $\gamma$ and $g(\mathbf{x})$ for those above — and proposes the next trial by maximising the ratio

$$
\mathbf{x}_{\text{next}} \;=\; \arg\max_{\mathbf{x}}\; \frac{l(\mathbf{x})}{g(\mathbf{x})}.
$$

The objective is the best validation loss of a trained trial, so **direction must be `minimize`**. The `MedianPruner` terminates a trial at step $t$ when its reported intermediate value exceeds the median of completed trials' values at step $t$, but only after `n_startup_trials` completed trials and `n_warmup_steps` reported steps.

> **What you should see:** a `Study` whose `direction` is `StudyDirection.MINIMIZE`, whose sampler is a `TPESampler` and whose pruner is a `MedianPruner`. The pruner's `_n_startup_trials` equals `phase_config.pruner_n_startup_trials` (8) and `_n_warmup_steps` equals `phase_config.pruner_n_warmup_steps` (8). The study starts empty (`len(study.trials) == 0`). In production the same study is loaded from a SQLite file by `tuning_configuration.n_gpus` (4) workers via `load_if_exists=True`.


In [ ]:
phase1_phase_config = tuning_configuration.phase1

phase1_study_pruner = optuna.pruners.MedianPruner(
    n_startup_trials = phase1_phase_config.pruner_n_startup_trials,
    n_warmup_steps   = phase1_phase_config.pruner_n_warmup_steps,
)

phase1_study_sampler = optuna.samplers.TPESampler(
    n_startup_trials = phase1_phase_config.pruner_n_startup_trials,
    multivariate     = True,
    constant_liar    = True,
    seed             = 42,
)

phase1_study = optuna.create_study(
    study_name = "inspect_unet_phase1",
    direction  = "minimize",
    sampler    = phase1_study_sampler,
    pruner     = phase1_study_pruner,
)

In [ ]:
print("study direction        :", phase1_study.direction)
print("sampler type           :", type(phase1_study.sampler).__name__)
print("pruner type            :", type(phase1_study.pruner).__name__)
print()
print("pruner n_startup_trials :", phase1_study.pruner._n_startup_trials)
print("pruner n_warmup_steps   :", phase1_study.pruner._n_warmup_steps)
print("config n_startup_trials :", phase1_phase_config.pruner_n_startup_trials)
print("config n_warmup_steps   :", phase1_phase_config.pruner_n_warmup_steps)
print()
print("sampler seed            :", phase1_study.sampler._seed)
print("trials at creation      :", len(phase1_study.trials))
print()
print("production storage URL pattern : sqlite:///<run_dir>/optuna.db   (load_if_exists=True)")
print("workers sharing the study      :", tuning_configuration.n_gpus)


In [ ]:
figure, (axis_left, axis_right) = plt.subplots(1, 2, figsize=(9.5, 3.4))

direction_label = phase1_study.direction.name
component_labels = ["direction\n(minimize)", "sampler\n(TPE)", "pruner\n(Median)"]
component_ok     = [
    direction_label == "MINIMIZE",
    type(phase1_study.sampler).__name__  == "TPESampler",
    type(phase1_study.pruner).__name__   == "MedianPruner",
]
axis_left.bar(component_labels, [1, 1, 1], color=[palette_phase1 if ok else palette_pruned for ok in component_ok], edgecolor="black", linewidth=0.6)
axis_left.set_ylim(0, 1.25)
axis_left.set_yticks([0, 1])
axis_left.set_ylabel("configured (1) / wrong (0)")
axis_left.set_title("Stage 1 — Study: core components")

pruner_labels = ["n_startup_trials", "n_warmup_steps"]
pruner_config = [phase1_phase_config.pruner_n_startup_trials, phase1_phase_config.pruner_n_warmup_steps]
pruner_actual = [phase1_study.pruner._n_startup_trials, phase1_study.pruner._n_warmup_steps]
bar_x = np.arange(len(pruner_labels))
axis_right.bar(bar_x - 0.18, pruner_config, width=0.36, label="config", color=palette_neutral, edgecolor="black", linewidth=0.6)
axis_right.bar(bar_x + 0.18, pruner_actual, width=0.36, label="study pruner", color=palette_phase1, edgecolor="black", linewidth=0.6)
axis_right.set_xticks(bar_x)
axis_right.set_xticklabels(pruner_labels)
axis_right.set_ylabel("count (trials / steps)")
axis_right.set_title("Stage 1 — Study: pruner matches config")
axis_right.legend(frameon=False)

figure.tight_layout()
figure.savefig(figure_directory / "stage_1_study_creation.pdf")
plt.show()


In [ ]:
stage_1_checks = [
    ("Study direction is minimize",            phase1_study.direction == optuna.study.StudyDirection.MINIMIZE),
    ("Sampler is TPESampler",                  type(phase1_study.sampler).__name__ == "TPESampler"),
    ("Pruner is MedianPruner",                 type(phase1_study.pruner).__name__  == "MedianPruner"),
    ("Pruner n_startup matches config",        phase1_study.pruner._n_startup_trials == phase1_phase_config.pruner_n_startup_trials),
    ("Pruner n_warmup matches config",         phase1_study.pruner._n_warmup_steps   == phase1_phase_config.pruner_n_warmup_steps),
    ("Sampler is seeded (determinism)",        phase1_study.sampler._seed is not None),
    ("Study starts empty",                     len(phase1_study.trials) == 0),
]
for stage_1_description, stage_1_condition in stage_1_checks:
    print(f"[{'PASS' if stage_1_condition else 'FAIL'}] {stage_1_description}")


### Common Mistakes — Stage 1: Study creation

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Tuner converges to the **worst** model | `direction` flipped to `"maximize"` while the objective returns a loss to be minimised | Print `study.direction`; it must be `MINIMIZE` because `_objective` returns `best_val_loss` |
| Four GPU workers explore identical trials | Each worker created its own in-memory study instead of loading the shared SQLite study | Confirm the storage URL is a `sqlite:///...optuna.db` and `load_if_exists=True`; check all workers share one `study_name` |
| Runs are not reproducible between launches | `TPESampler` left unseeded | Print `study.sampler._seed`; the production sampler fixes `seed=42` |
| Almost every trial is pruned immediately | `MedianPruner` warm-up too short relative to per-epoch noise, or `n_startup_trials` too small | Compare `pruner._n_startup_trials` / `_n_warmup_steps` against `phase_config`; raise if curriculum/warm-up dominates early epochs |
| Pruner never fires | `n_warmup_steps` >= `n_epochs`, so no trial is ever evaluated past warm-up | Check `pruner._n_warmup_steps` against `phase_config.n_epochs` |

**Passing to Stage 2:** `phase1_study` — an empty `optuna.Study` (minimize, TPESampler, MedianPruner) whose trials will be generated by sampling the Phase-1 search space.


---
## Stage 2: Parameter sampling — Phase-1 search space

**Callable:** `ParamSampler.sample(trial, space)`
**Input:** an `optuna.Trial` and the search-space dictionary `model_config_cls.tunable_lr_params()`.
**Output:** a `dict` mapping each hyperparameter name to a concretely sampled value, registered on the trial.

`ParamSampler` dispatches on each entry's `"type"`:

- `"float"`: `trial.suggest_float(name, low, high, log)` — for learning rates and weight decays the range is sampled on a **log scale**, so the prior is uniform in $\log_{10}(\text{value})$ over $[\log_{10}\text{low}, \log_{10}\text{high}]$.
- `"categorical"`: `trial.suggest_categorical(name, choices)`.
- `"indexed_categorical"`: suggests an integer index `name + "__idx"` over `range(len(choices))` and returns `choices[idx]` — used when a choice is itself a list (e.g. feature widths) that Optuna cannot store directly.

For Phase 1 the space is `tunable_lr_params()`: eight log-uniform float ranges (`*_lr` in $[10^{-5}, 10^{-2}]$, `*_wd` in $[10^{-6}, 10^{-1}]$) plus a linear `dropout` in $[0, 0.5]$.

> **What you should see:** a sampled `dict` with exactly the keys of `tunable_lr_params()` (9 for the UNet config). Every `*_lr` value lies in $[10^{-5}, 10^{-2}]$, every `*_wd` value in $[10^{-6}, 10^{-1}]$, and `dropout` in $[0, 0.5]$. Sampling many trials should fill the log-scaled ranges roughly uniformly in log-space, not pile up near the upper bound.


In [ ]:
phase1_search_space = model_config_class.tunable_lr_params()

sampling_study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
parameter_sampler = ParamSampler()

number_of_sampling_trials = 256
phase1_sampled_records = []
for _ in range(number_of_sampling_trials):
    sampling_trial = sampling_study.ask()
    sampled_values = parameter_sampler.sample(sampling_trial, phase1_search_space)
    sampling_study.tell(sampling_trial, 0.0)
    phase1_sampled_records.append(sampled_values)

single_phase1_sample = phase1_sampled_records[0]


In [ ]:
print("Phase-1 search space keys :", list(phase1_search_space.keys()))
print("number of sampled params  :", len(single_phase1_sample))
print()
print("one sampled configuration:")
for parameter_name, parameter_value in single_phase1_sample.items():
    specification = phase1_search_space[parameter_name]
    print(f"  {parameter_name:<16}: {parameter_value:<24} (type={specification['type']}, log={specification.get('log', False)})")
print()
float_parameter_names = [name for name, spec in phase1_search_space.items() if spec["type"] == "float"]
print("per-parameter min / max over", number_of_sampling_trials, "trials:")
for parameter_name in float_parameter_names:
    column = np.array([record[parameter_name] for record in phase1_sampled_records])
    print(f"  {parameter_name:<16}: min={column.min():.3e}  max={column.max():.3e}  spec=[{phase1_search_space[parameter_name]['low']:.0e}, {phase1_search_space[parameter_name]['high']:.0e}]")


In [ ]:
log_float_names = [name for name in float_parameter_names if phase1_search_space[name].get("log", False)]
number_of_panels = len(log_float_names) + 1
number_of_columns = 3
number_of_rows = int(np.ceil(number_of_panels / number_of_columns))

figure, axes = plt.subplots(number_of_rows, number_of_columns, figsize=(11, 2.9 * number_of_rows))
axes = np.atleast_1d(axes).ravel()

for panel_index, parameter_name in enumerate(log_float_names):
    column = np.array([record[parameter_name] for record in phase1_sampled_records])
    axis = axes[panel_index]
    axis.hist(np.log10(column), bins=24, color=palette_phase1, edgecolor="black", linewidth=0.4)
    axis.axvline(np.log10(phase1_search_space[parameter_name]["low"]),  color=palette_pruned, linestyle="--", linewidth=1.0)
    axis.axvline(np.log10(phase1_search_space[parameter_name]["high"]), color=palette_pruned, linestyle="--", linewidth=1.0)
    axis.set_xlabel(f"log10({parameter_name})")
    axis.set_ylabel("trial count")
    axis.set_title(parameter_name)

dropout_column = np.array([record["dropout"] for record in phase1_sampled_records])
dropout_axis = axes[len(log_float_names)]
dropout_axis.hist(dropout_column, bins=24, color=palette_neutral, edgecolor="black", linewidth=0.4)
dropout_axis.axvline(phase1_search_space["dropout"]["low"],  color=palette_pruned, linestyle="--", linewidth=1.0)
dropout_axis.axvline(phase1_search_space["dropout"]["high"], color=palette_pruned, linestyle="--", linewidth=1.0)
dropout_axis.set_xlabel("dropout (probability)")
dropout_axis.set_ylabel("trial count")
dropout_axis.set_title("dropout (linear)")

for spare_index in range(number_of_panels, len(axes)):
    axes[spare_index].axis("off")

figure.suptitle("Stage 2 — Parameter sampling: Phase-1 search-space coverage", y=1.02)
figure.tight_layout()
figure.savefig(figure_directory / "stage_2_param_sampling.pdf")
plt.show()


In [ ]:
within_lr_bounds = all(
    phase1_search_space[name]["low"] <= record[name] <= phase1_search_space[name]["high"]
    for record in phase1_sampled_records
    for name in float_parameter_names
)
sampled_keys = set(single_phase1_sample.keys())
expected_keys = set(phase1_search_space.keys())
dropout_in_range = bool(np.all((dropout_column >= phase1_search_space["dropout"]["low"]) & (dropout_column <= phase1_search_space["dropout"]["high"])))

stage_2_checks = [
    ("Sampled keys match search space",        sampled_keys == expected_keys),
    ("Sampled dict length matches space",      len(single_phase1_sample) == len(phase1_search_space)),
    ("All float samples within bounds",        within_lr_bounds),
    ("dropout within [low, high]",             dropout_in_range),
    ("No NaN among sampled values",            not any(np.isnan(v) for record in phase1_sampled_records for v in record.values())),
    ("Search space is non-empty",              len(phase1_search_space) > 0),
]
for stage_2_description, stage_2_condition in stage_2_checks:
    print(f"[{'PASS' if stage_2_condition else 'FAIL'}] {stage_2_description}")


### Common Mistakes — Stage 2: Parameter sampling

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Learning rates cluster near the upper bound | `"log": True` dropped from the spec, so sampling is linear and over-weights large values | Histogram `log10(value)`; a log-uniform prior is flat in log-space, a linear one is skewed |
| `features` choice raises a serialization error | A list-valued choice declared as `"categorical"` instead of `"indexed_categorical"` | Inspect the spec `type`; list choices must go through the `__idx` indexed path |
| Phase-2 re-tunes a learning rate already fixed by Phase 1 | Overlapping keys between `tunable_lr_params()` and `tunable_arch_params()` | Intersect the two key sets — they must be disjoint (checked in Stage 7) |
| Sampled value out of the documented range | `low`/`high` in the spec inconsistent with the dataclass default field bounds | Compare `spec['low']`/`spec['high']` against the config field domain |
| Identical samples across workers | Sampler seed shared without `constant_liar`, so concurrent asks collide | Confirm `constant_liar=True` in the production sampler |

**Passing to Stage 3:** `single_phase1_sample` — a `dict` of sampled Phase-1 hyperparameters that `_apply_params` will `setattr` onto a fresh model config before a trial trains.


---
## Stage 3: Trial pipeline assembly — per-trial config isolation

**Callable:** `BaseTuner._objective(trial)` (config-assembly portion) building a `TrialPipeline`
**Input:** an `optuna.Trial`.
**Output:** a fresh model config with sampled params applied, a **deep-copied** trainer config and dataset config with per-trial overrides, and a `TrialPipeline` whose seed and run name are derived from `trial.number`.

For each trial the objective:

1. Instantiates a fresh `model_config_cls()` and applies the sampled params via `_apply_params` (Phase 1: learning-dynamics; Phase 2: architecture + frozen Phase-1 best).
2. `copy.deepcopy`s the base trainer and dataset configs so no trial mutates shared state.
3. Overrides `training.epochs`, `scheduler.epochs` (both to `tune_cfg.n_epochs`), `early_stopping.patience` (to `tune_cfg.early_stop_patience`) and `io.logdir` (to `<log_dir>/trial_<number:04d>`).
4. Builds a `TrialPipeline(seed=trial.number, run_name=f"{run_name_prefix}{trial.number:04d}", trial=trial)`.

> **What you should see:** the per-trial trainer config has `training.epochs == scheduler.epochs == tune_cfg.n_epochs` (30) and `early_stopping.patience == tune_cfg.early_stop_patience` (8 in Phase 1). The deep copy is a genuinely independent object (mutating it must not touch the base config). The per-trial log directory ends with `trial_0000` for trial number 0, and the seed equals `trial.number`.


In [ ]:
class ConfigAssemblyProbe(BaseTuner):
    run_name_prefix = "phase1_trial_"

    def _apply_params(self, trial, model_config):
        sampled = self.sampler.sample(trial, self.model_config_cls.tunable_lr_params())
        for parameter_name, parameter_value in sampled.items():
            setattr(model_config, parameter_name, parameter_value)

    def assemble(self, trial):
        model_config = self.model_config_cls()
        self._apply_params(trial, model_config)

        trainer_config = copy.deepcopy(self.base_trainer_config)
        dataset_config = copy.deepcopy(self.base_dataset_config)

        trainer_config["training"]["epochs"]         = self.tune_cfg.n_epochs
        trainer_config["scheduler"]["epochs"]        = self.tune_cfg.n_epochs
        trainer_config["early_stopping"]["patience"] = self.tune_cfg.early_stop_patience
        trainer_config["io"]["logdir"]               = str(Path(self.log_dir) / f"trial_{trial.number:04d}")

        return model_config, trainer_config, dataset_config


base_trainer_config_surrogate = {
    "training"       : {"epochs": 60},
    "scheduler"      : {"epochs": 60},
    "early_stopping" : {"patience": 8},
    "io"             : {"logdir": "/base/logdir"},
}
base_dataset_config_surrogate = {"batch_size": 256, "num_workers": 4}

config_assembly_probe = ConfigAssemblyProbe(
    model_name          = model_name_under_inspection,
    model_config_cls    = model_config_class,
    base_trainer_config = base_trainer_config_surrogate,
    base_dataset_config = base_dataset_config_surrogate,
    tune_cfg            = tuning_configuration.phase1,
    log_dir             = "/inspect/phase1",
    logger              = None,
)

assembly_study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=7))
assembly_trial = assembly_study.ask()
assembled_model_config, assembled_trainer_config, assembled_dataset_config = config_assembly_probe.assemble(assembly_trial)


In [ ]:
print("trial number               :", assembly_trial.number)
print("per-trial log dir          :", assembled_trainer_config["io"]["logdir"])
print()
print("trainer training.epochs    :", assembled_trainer_config["training"]["epochs"],  "  (tune_cfg.n_epochs =", tuning_configuration.phase1.n_epochs, ")")
print("trainer scheduler.epochs   :", assembled_trainer_config["scheduler"]["epochs"], "  (tune_cfg.n_epochs =", tuning_configuration.phase1.n_epochs, ")")
print("trainer early_stop patience:", assembled_trainer_config["early_stopping"]["patience"], "  (tune_cfg.early_stop_patience =", tuning_configuration.phase1.early_stop_patience, ")")
print()
print("base config still intact   :")
print("  base training.epochs     :", base_trainer_config_surrogate["training"]["epochs"], " (deep copy did not mutate base)")
print("  base io.logdir           :", base_trainer_config_surrogate["io"]["logdir"])
print()
print("applied learning-rate params on fresh model config:")
for parameter_name in model_config_class.tunable_lr_params():
    print(f"  {parameter_name:<16}: {getattr(assembled_model_config, parameter_name)}")


In [ ]:
override_labels = ["training.epochs", "scheduler.epochs", "early_stop.patience"]
override_actual = [
    assembled_trainer_config["training"]["epochs"],
    assembled_trainer_config["scheduler"]["epochs"],
    assembled_trainer_config["early_stopping"]["patience"],
]
override_expected = [
    tuning_configuration.phase1.n_epochs,
    tuning_configuration.phase1.n_epochs,
    tuning_configuration.phase1.early_stop_patience,
]
base_values = [
    base_trainer_config_surrogate["training"]["epochs"],
    base_trainer_config_surrogate["scheduler"]["epochs"],
    base_trainer_config_surrogate["early_stopping"]["patience"],
]

bar_x = np.arange(len(override_labels))
figure, axis = plt.subplots(figsize=(8.5, 3.6))
axis.bar(bar_x - 0.27, base_values,     width=0.27, label="base config", color=palette_neutral, edgecolor="black", linewidth=0.6)
axis.bar(bar_x,        override_actual,  width=0.27, label="assembled trial config", color=palette_phase1, edgecolor="black", linewidth=0.6)
axis.bar(bar_x + 0.27, override_expected, width=0.27, label="expected (tune_cfg)", color=palette_phase2, edgecolor="black", linewidth=0.6)
axis.set_xticks(bar_x)
axis.set_xticklabels(override_labels)
axis.set_ylabel("epochs / patience (count)")
axis.set_title("Stage 3 — Trial assembly: per-trial overrides vs base config")
axis.legend(frameon=False)

figure.tight_layout()
figure.savefig(figure_directory / "stage_3_trial_assembly.pdf")
plt.show()


In [ ]:
trial_log_directory = Path(assembled_trainer_config["io"]["logdir"])

stage_3_checks = [
    ("training.epochs == tune_cfg.n_epochs",        assembled_trainer_config["training"]["epochs"]  == tuning_configuration.phase1.n_epochs),
    ("scheduler.epochs == tune_cfg.n_epochs",       assembled_trainer_config["scheduler"]["epochs"] == tuning_configuration.phase1.n_epochs),
    ("early_stop patience == tune_cfg",             assembled_trainer_config["early_stopping"]["patience"] == tuning_configuration.phase1.early_stop_patience),
    ("per-trial log dir uses 4-digit number",       trial_log_directory.name == f"trial_{assembly_trial.number:04d}"),
    ("deep copy left base epochs untouched",        base_trainer_config_surrogate["training"]["epochs"] == 60),
    ("deep copy left base logdir untouched",        base_trainer_config_surrogate["io"]["logdir"] == "/base/logdir"),
    ("fresh model config carries sampled lr",       getattr(assembled_model_config, "encoder_lr") != UNetConfig().encoder_lr),
]
for stage_3_description, stage_3_condition in stage_3_checks:
    print(f"[{'PASS' if stage_3_condition else 'FAIL'}] {stage_3_description}")


### Common Mistakes — Stage 3: Trial pipeline assembly

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Trial *N* inherits trial *N-1*'s hyperparameters | Base config mutated in place instead of `copy.deepcopy`'d | Mutate the assembled config and assert the base is unchanged (done here) |
| Scheduler never completes its cosine cycle | `scheduler.epochs` left at the long base value while `training.epochs` set to `n_epochs` | Print both; they must both equal `tune_cfg.n_epochs` |
| Trials overwrite each other's logs/checkpoints | `io.logdir` not made trial-specific | Check the log dir ends with `trial_<number:04d>` |
| Two trials produce identical curves | Seed not derived from `trial.number`, so every trial uses the same RNG stream | `TrialPipeline` is built with `seed=trial.number` — confirm it varies |
| Phase-2 trial ignores frozen Phase-1 params | `_apply_params` sets arch params but forgets to copy `best_phase1_params` | Inspect the model config after assembly for the Phase-1 keys (Stage 7) |

**Passing to Stage 4:** `assembled_trainer_config` — a deep-copied, trial-scoped trainer config; the `TrialPipeline` built from it trains and reports to Optuna via the trial callback inspected next.


---
## Stage 4: Trial trainer — per-epoch reporting and pruning

**Callable:** `TrialTrainer._trial_callback(val_loss, epoch)`
**Input:** the validation loss and epoch index emitted by the training loop each epoch.
**Output:** none directly; it calls `trial.report(val_loss, epoch)` and raises `optuna.exceptions.TrialPruned()` when `trial.should_prune()` returns `True`.

`TrialTrainer` subclasses the training-pipeline `Trainer` and overrides only the per-epoch hook. After each validation pass it streams the intermediate value into the study, where the `MedianPruner` compares it to the running median at the same step. A trial that lags behind is stopped early to spend the budget on promising regions:

$$
\text{prune at step } t \iff v_t \;>\; \operatorname{median}\big(\{v_t^{(j)} : \text{completed trial } j\}\big), \quad t \ge \texttt{n\_warmup\_steps},\; \#\text{completed} \ge \texttt{n\_startup\_trials}.
$$

> **What you should see:** with a study warmed up by enough completed trials, a trial whose loss curve sits above the median is pruned **before** reaching `n_epochs`, raising `TrialPruned`; a trial whose curve tracks the best completed trials runs to completion. Pruned trials have `state == TrialState.PRUNED` and fewer reported steps than `n_epochs`.


In [ ]:
pruning_study = optuna.create_study(
    direction = "minimize",
    sampler   = optuna.samplers.TPESampler(seed=42),
    pruner    = optuna.pruners.MedianPruner(
        n_startup_trials = tuning_configuration.phase1.pruner_n_startup_trials,
        n_warmup_steps   = tuning_configuration.phase1.pruner_n_warmup_steps,
    ),
)

number_of_epochs_per_trial = tuning_configuration.phase1.n_epochs
random_generator = np.random.default_rng(0)


def surrogate_loss_curve(asymptote, noise_scale):
    epochs = np.arange(number_of_epochs_per_trial)
    decay  = 0.6 * np.exp(-epochs / 6.0)
    noise  = random_generator.normal(0.0, noise_scale, size=number_of_epochs_per_trial)
    return asymptote + decay + noise


class TrialReportingProbe:
    def __init__(self, study):
        self.study = study

    def run_trial(self, asymptote, noise_scale):
        trial = self.study.ask()
        loss_curve = surrogate_loss_curve(asymptote, noise_scale)
        pruned_at = None
        for epoch_index in range(number_of_epochs_per_trial):
            value = float(loss_curve[epoch_index])
            trial.report(value, epoch_index)
            if trial.should_prune():
                pruned_at = epoch_index
                self.study.tell(trial, state=optuna.trial.TrialState.PRUNED)
                return trial.number, "PRUNED", pruned_at, loss_curve
        self.study.tell(trial, float(loss_curve[-1]))
        return trial.number, "COMPLETE", None, loss_curve


reporting_probe = TrialReportingProbe(pruning_study)

trial_outcomes = []
for asymptote_value in np.linspace(0.10, 0.16, tuning_configuration.phase1.pruner_n_startup_trials):
    trial_outcomes.append(reporting_probe.run_trial(asymptote_value, 0.01))

good_trial_outcome = reporting_probe.run_trial(0.08, 0.01)
bad_trial_outcome  = reporting_probe.run_trial(0.40, 0.01)
trial_outcomes.extend([good_trial_outcome, bad_trial_outcome])


In [ ]:
state_counts = {state.name: 0 for state in [optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED]}
for completed_trial in pruning_study.trials:
    state_counts[completed_trial.state.name] = state_counts.get(completed_trial.state.name, 0) + 1

print("total trials run           :", len(pruning_study.trials))
print("complete trials            :", state_counts.get("COMPLETE", 0))
print("pruned trials              :", state_counts.get("PRUNED", 0))
print()
good_number, good_state, good_pruned_at, _ = good_trial_outcome
bad_number,  bad_state,  bad_pruned_at,  _ = bad_trial_outcome
print(f"low-loss trial  #{good_number} : state={good_state}, pruned_at={good_pruned_at}")
print(f"high-loss trial #{bad_number} : state={bad_state}, pruned_at={bad_pruned_at}")
print()
print("epochs budget per trial    :", number_of_epochs_per_trial)
print("pruner warm-up steps       :", tuning_configuration.phase1.pruner_n_warmup_steps)
print("pruner startup trials      :", tuning_configuration.phase1.pruner_n_startup_trials)


In [ ]:
figure, (axis_curves, axis_counts) = plt.subplots(1, 2, figsize=(10.5, 3.8))

epochs_axis = np.arange(number_of_epochs_per_trial)
for outcome in trial_outcomes[:tuning_configuration.phase1.pruner_n_startup_trials]:
    _, _, _, loss_curve = outcome
    axis_curves.plot(epochs_axis, loss_curve, color=palette_neutral, alpha=0.35, linewidth=0.9)

_, _, _, good_curve = good_trial_outcome
_, _, bad_pruned_at, bad_curve = bad_trial_outcome
axis_curves.plot(epochs_axis, good_curve, color=palette_phase1, linewidth=1.8, label="completed (low loss)")
if bad_pruned_at is not None:
    axis_curves.plot(epochs_axis[:bad_pruned_at + 1], bad_curve[:bad_pruned_at + 1], color=palette_pruned, linewidth=1.8, label="pruned (high loss)")
    axis_curves.scatter([bad_pruned_at], [bad_curve[bad_pruned_at]], color=palette_pruned, zorder=5, marker="x", s=60)
axis_curves.axvline(tuning_configuration.phase1.pruner_n_warmup_steps, color="black", linestyle=":", linewidth=1.0, label="warm-up boundary")
axis_curves.set_xlabel("epoch (index)")
axis_curves.set_ylabel("validation loss (dimensionless)")
axis_curves.set_title("Stage 4 — Trial trainer: reported curves and pruning")
axis_curves.legend(frameon=False)

state_labels = ["COMPLETE", "PRUNED"]
state_values = [state_counts.get("COMPLETE", 0), state_counts.get("PRUNED", 0)]
axis_counts.bar(state_labels, state_values, color=[palette_phase1, palette_pruned], edgecolor="black", linewidth=0.6)
axis_counts.set_ylabel("trial count")
axis_counts.set_title("Stage 4 — Trial trainer: complete vs pruned")

figure.tight_layout()
figure.savefig(figure_directory / "stage_4_trial_reporting.pdf")
plt.show()


In [ ]:
bad_trial_object  = pruning_study.trials[bad_number]
good_trial_object = pruning_study.trials[good_number]

stage_4_checks = [
    ("High-loss trial was pruned",               bad_trial_object.state == optuna.trial.TrialState.PRUNED),
    ("Low-loss trial completed",                 good_trial_object.state == optuna.trial.TrialState.COMPLETE),
    ("Pruned trial stopped before budget",       bad_pruned_at is not None and bad_pruned_at < number_of_epochs_per_trial - 1),
    ("Pruning only after warm-up steps",         bad_pruned_at is None or bad_pruned_at >= tuning_configuration.phase1.pruner_n_warmup_steps),
    ("Study contains pruned trials",             state_counts.get("PRUNED", 0) > 0),
    ("Reported steps recorded on pruned trial",  len(bad_trial_object.intermediate_values) > 0),
]
for stage_4_description, stage_4_condition in stage_4_checks:
    print(f"[{'PASS' if stage_4_condition else 'FAIL'}] {stage_4_description}")


### Common Mistakes — Stage 4: Trial trainer reporting and pruning

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Nothing is ever pruned | `_trial_callback` not wired into the validation loop, so `trial.report` never runs | Check `intermediate_values` is non-empty for in-progress trials |
| Promising trials pruned at epoch 0 | `n_warmup_steps` too small relative to warm-up / curriculum, which inflates early-epoch loss | Compare the prune epoch against `pruner_n_warmup_steps`; align it past the LR warm-up |
| `TrialPruned` swallowed and counted as a failure | `except Exception` in `_objective` catches `TrialPruned` before the dedicated `except` | Confirm the `except optuna.exceptions.TrialPruned: raise` clause precedes the generic handler |
| Pruner compares against the wrong step | Epoch index passed to `report` is off by one or reset each epoch | Print the keys of `intermediate_values`; they must be the monotone epoch indices |
| Loss reported is train loss, not val loss | Callback fed the wrong scalar | The callback signature is `(val_loss, epoch)` — verify the source is the validation pass |

**Passing to Stage 5:** the trial-reporting contract verified here is what `study.optimize` invokes for every trial in the Phase-1 loop, where pruned and completed trials accumulate in the study.


---
## Stage 5: Phase-1 tuning loop — learning-dynamics search

**Callable:** `Phase1Tuner.run(study, n_trials)` (delegating to `study.optimize(self._objective, n_trials)`)
**Input:** the Phase-1 `Study` and the worker's trial budget.
**Output:** the same `Study`, now populated with up to `n_trials` complete/pruned trials and exposing `study.best_trial`.

`Phase1Tuner._apply_params` samples `tunable_lr_params()` and `setattr`s them onto a fresh model config; the objective trains the trial and returns the best validation loss. Across the four GPU workers the trial budget is split by `_distribute_trials(total, n_workers)`, so the four workers collectively run `phase1.n_trials` trials against the shared study.

Here we run the **identical** sampler/pruner over a closed-form surrogate objective (a smooth bowl in log-LR with a dropout penalty) so the full loop, the best-trial selection and the optimisation trajectory are all exercised without a GPU.

> **What you should see:** a study with `n_trials` finished trials, a non-empty set of `COMPLETE` trials, and a `best_value` no larger than any individual completed trial's value. The running best (objective vs trial index) should be **monotonically non-increasing**. With `multivariate` TPE the later trials should concentrate near the surrogate optimum.


In [ ]:
phase1_loop_search_space = model_config_class.tunable_lr_params()
phase1_loop_trial_budget = 80


def phase1_surrogate_objective(trial):
    sampled = parameter_sampler.sample(trial, phase1_loop_search_space)
    log_encoder_lr = np.log10(sampled["encoder_lr"])
    log_decoder_lr = np.log10(sampled["decoder_lr"])
    log_encoder_wd = np.log10(sampled["encoder_wd"])
    dropout_value  = sampled["dropout"]
    loss = (
        0.12
        + 0.35 * (log_encoder_lr + 3.2) ** 2
        + 0.20 * (log_decoder_lr + 3.0) ** 2
        + 0.05 * (log_encoder_wd + 4.0) ** 2
        + 0.40 * (dropout_value - 0.10) ** 2
    )
    return float(loss)


phase1_loop_study = optuna.create_study(
    direction = "minimize",
    sampler   = optuna.samplers.TPESampler(seed=42, multivariate=True, constant_liar=True),
    pruner    = optuna.pruners.MedianPruner(
        n_startup_trials = tuning_configuration.phase1.pruner_n_startup_trials,
        n_warmup_steps   = tuning_configuration.phase1.pruner_n_warmup_steps,
    ),
)
phase1_loop_study.optimize(phase1_surrogate_objective, n_trials=phase1_loop_trial_budget)


In [ ]:
phase1_completed_trials = [t for t in phase1_loop_study.trials if t.state == optuna.trial.TrialState.COMPLETE]
phase1_objective_values = np.array([t.value for t in phase1_completed_trials])

print("trials requested           :", phase1_loop_trial_budget)
print("trials recorded            :", len(phase1_loop_study.trials))
print("completed trials           :", len(phase1_completed_trials))
print()
print("best value                 :", phase1_loop_study.best_value)
print("best trial number          :", phase1_loop_study.best_trial.number)
print("objective min / mean / max :", f"{phase1_objective_values.min():.4f} / {phase1_objective_values.mean():.4f} / {phase1_objective_values.max():.4f}")
print()
print("best Phase-1 raw params:")
for parameter_name, parameter_value in phase1_loop_study.best_trial.params.items():
    print(f"  {parameter_name:<16}: {parameter_value}")


In [ ]:
trial_indices = np.array([t.number for t in phase1_completed_trials])
running_best  = np.minimum.accumulate(phase1_objective_values)

figure, (axis_history, axis_scatter) = plt.subplots(1, 2, figsize=(10.5, 3.8))

axis_history.scatter(trial_indices, phase1_objective_values, color=palette_neutral, s=18, alpha=0.6, label="trial objective")
axis_history.plot(trial_indices, running_best, color=palette_phase1, linewidth=1.8, label="running best")
axis_history.set_xlabel("trial index")
axis_history.set_ylabel("best validation loss (surrogate)")
axis_history.set_title("Stage 5 — Phase 1: objective vs trial")
axis_history.legend(frameon=False)

log_encoder_lr_values = np.array([np.log10(t.params["encoder_lr"]) for t in phase1_completed_trials])
log_decoder_lr_values = np.array([np.log10(t.params["decoder_lr"]) for t in phase1_completed_trials])
scatter = axis_scatter.scatter(log_encoder_lr_values, log_decoder_lr_values, c=phase1_objective_values, cmap="viridis", s=24, edgecolor="black", linewidth=0.3)
axis_scatter.scatter([np.log10(phase1_loop_study.best_trial.params["encoder_lr"])], [np.log10(phase1_loop_study.best_trial.params["decoder_lr"])], color=palette_pruned, marker="*", s=180, edgecolor="black", linewidth=0.5, label="best trial", zorder=5)
axis_scatter.set_xlabel("log10(encoder_lr)")
axis_scatter.set_ylabel("log10(decoder_lr)")
axis_scatter.set_title("Stage 5 — Phase 1: explored learning-rate plane")
axis_scatter.legend(frameon=False)
colorbar = figure.colorbar(scatter, ax=axis_scatter)
colorbar.set_label("validation loss (surrogate)")

figure.tight_layout()
figure.savefig(figure_directory / "stage_5_phase1_loop.pdf")
plt.show()


In [ ]:
running_best_is_monotone = bool(np.all(np.diff(running_best) <= 1e-12))

stage_5_checks = [
    ("Recorded trials == requested budget",      len(phase1_loop_study.trials) == phase1_loop_trial_budget),
    ("At least one completed trial",             len(phase1_completed_trials) > 0),
    ("best_value <= all completed values",       phase1_loop_study.best_value <= phase1_objective_values.max() + 1e-9),
    ("best_value equals min completed value",    abs(phase1_loop_study.best_value - phase1_objective_values.min()) < 1e-9),
    ("Running best is non-increasing",           running_best_is_monotone),
    ("best_trial params keyed by search space",  set(phase1_loop_study.best_trial.params.keys()) == set(phase1_loop_search_space.keys())),
]
for stage_5_description, stage_5_condition in stage_5_checks:
    print(f"[{'PASS' if stage_5_condition else 'FAIL'}] {stage_5_description}")


### Common Mistakes — Stage 5: Phase-1 tuning loop

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Running-best curve goes **up** | Direction misconfigured or objective sign flipped | Plot `np.minimum.accumulate(values)`; it must be non-increasing for a minimise study |
| Fewer trials than `n_trials` budget | A worker crashed, or all trials raised and were turned into pruned/failed states | Compare `len(study.trials)` against the per-worker `_distribute_trials` allocation |
| TPE never exploits the good region | `n_trials` too small to leave the `n_startup_trials` random phase | Ensure `n_trials` >> `pruner_n_startup_trials`; inspect late-trial concentration in the LR plane |
| Best trial sits on a search-space boundary | The range is too narrow and clips the optimum | Check whether `best_trial.params` hit `low`/`high` of the spec |
| Workers double-count or overwrite trials | Study object shared incorrectly (in-memory copy per worker) | All workers must `load_study` the same SQLite `study_name` |

**Passing to Stage 6:** `phase1_loop_study` — a finished Phase-1 study whose `best_trial` holds the learning-dynamics hyperparameters to be decoded and frozen for Phase 2.


---
## Stage 6: Best Phase-1 parameter extraction and decoding

**Callable:** `study.best_trial` + `TuningOrchestrator._decode_phase1_best(model_name, best_p1_trial)`
**Input:** the completed Phase-1 `Study`.
**Output:** a decoded `dict` of the best Phase-1 hyperparameters, serialised to `phase1_best.json` and later loaded as `best_phase1_params` for Phase 2.

Optuna stores `indexed_categorical` choices as integer `name + "__idx"` keys (because list-valued choices cannot be stored directly). The decoder rebuilds the human-meaningful values: for each raw key ending in `__idx` it looks up the corresponding `indexed_categorical` spec in `tunable_lr_params()` and replaces the index with `choices[index]`; all other keys pass through unchanged.

> **What you should see:** a decoded `dict` whose keys are a subset of the Phase-1 search space (no `__idx` suffixes remain), values within the original spec ranges, and the loss equal to `study.best_value`. For Phase-1's purely float space the decoding is an identity on keys, but the decoder must still be exercised so it does not corrupt the seed handed to Phase 2.


In [ ]:
def decode_phase1_best(model_config_cls, best_trial):
    learning_rate_space = model_config_cls.tunable_lr_params()
    raw_parameters      = dict(best_trial.params)
    decoded_parameters  = {}
    for raw_key, raw_value in raw_parameters.items():
        if raw_key.endswith("__idx"):
            base_name     = raw_key[:-5]
            specification = learning_rate_space.get(base_name, {})
            if specification.get("type") == "indexed_categorical":
                decoded_parameters[base_name] = specification["choices"][raw_value]
        else:
            decoded_parameters[raw_key] = raw_value
    return decoded_parameters


best_phase1_trial    = phase1_loop_study.best_trial
decoded_phase1_params = decode_phase1_best(model_config_class, best_phase1_trial)


In [ ]:
print("best Phase-1 trial number  :", best_phase1_trial.number)
print("best Phase-1 value         :", best_phase1_trial.value)
print()
print("raw param keys             :", sorted(best_phase1_trial.params.keys()))
print("decoded param keys         :", sorted(decoded_phase1_params.keys()))
print("any __idx left in decoded  :", any(k.endswith("__idx") for k in decoded_phase1_params))
print()
print("decoded best Phase-1 params:")
phase1_search_space_keys = model_config_class.tunable_lr_params()
for parameter_name, parameter_value in decoded_phase1_params.items():
    specification = phase1_search_space_keys.get(parameter_name, {})
    print(f"  {parameter_name:<16}: {parameter_value:<24} (spec type={specification.get('type', 'n/a')})")


In [ ]:
decoded_names  = list(decoded_phase1_params.keys())
decoded_values = np.array([decoded_phase1_params[name] for name in decoded_names])

figure, axis = plt.subplots(figsize=(9.5, 3.8))
bar_positions = np.arange(len(decoded_names))
axis.bar(bar_positions, decoded_values, color=palette_phase1, edgecolor="black", linewidth=0.6, log=True)
axis.set_xticks(bar_positions)
axis.set_xticklabels(decoded_names, rotation=40, ha="right")
axis.set_ylabel("decoded value (log scale)")
axis.set_title(f"Stage 6 — Best Phase-1 params (trial {best_phase1_trial.number}, loss {best_phase1_trial.value:.4f})")

figure.tight_layout()
figure.savefig(figure_directory / "stage_6_phase1_best.pdf")
plt.show()


In [ ]:
no_index_suffix      = not any(name.endswith("__idx") for name in decoded_phase1_params)
keys_are_subset      = set(decoded_phase1_params.keys()).issubset(set(model_config_class.tunable_lr_params().keys()))
values_within_bounds = all(
    model_config_class.tunable_lr_params()[name]["low"] <= value <= model_config_class.tunable_lr_params()[name]["high"]
    for name, value in decoded_phase1_params.items()
    if model_config_class.tunable_lr_params()[name]["type"] == "float"
)

stage_6_checks = [
    ("best_trial value == best_value",           abs(best_phase1_trial.value - phase1_loop_study.best_value) < 1e-12),
    ("No __idx keys remain after decoding",      no_index_suffix),
    ("Decoded keys subset of Phase-1 space",     keys_are_subset),
    ("Decoded float values within bounds",       values_within_bounds),
    ("Decoded dict is non-empty",                len(decoded_phase1_params) > 0),
    ("Decoded params are JSON-serialisable",     all(isinstance(v, (int, float, str, list)) for v in decoded_phase1_params.values())),
]
for stage_6_description, stage_6_condition in stage_6_checks:
    print(f"[{'PASS' if stage_6_condition else 'FAIL'}] {stage_6_description}")


### Common Mistakes — Stage 6: Best Phase-1 extraction

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Phase 2 receives raw `__idx` integers | Decoder skipped, so the `indexed_categorical` index leaks as a hyperparameter | Assert no key ends with `__idx` in the decoded dict |
| `best_trial` raises `ValueError` | The study has zero completed trials (all pruned/failed) | Check `len([t for t in study.trials if t.state == COMPLETE]) > 0` |
| Phase-2 seed silently empty | `phase1_best.json` written before Phase 1 finished, or wrong path | Confirm the file exists and round-trips to the decoded dict |
| Best value disagrees with the curve | Reading `best_trial.params` from the wrong (e.g. last) trial | Cross-check `best_trial.value == study.best_value` |
| A decoded param is dropped | Spec lookup fails because the search space changed between sampling and decoding | The decode space must be the **same** `tunable_lr_params()` used to sample |

**Passing to Stage 7:** `decoded_phase1_params` — the frozen learning-dynamics hyperparameters that Phase 2 holds fixed while it searches the architecture space.


---
## Stage 7: Phase-2 tuning loop — architecture search with frozen Phase-1 best

**Callable:** `Phase2Tuner.run(study, n_trials)` with `best_phase1_params = decoded_phase1_params`
**Input:** a fresh Phase-2 `Study`, the worker budget, and the frozen Phase-1 best params.
**Output:** the Phase-2 `Study` populated with trials over `tunable_arch_params()`, the Phase-1 best held fixed on every trial's model config.

`Phase2Tuner._apply_params` samples **only** the architectural search space (feature widths via `indexed_categorical`, bottleneck factor, activation, normalization, upsampling), then writes the frozen Phase-1 best params onto the same model config (skipping any attribute the config does not expose). The two phases therefore optimise **disjoint** coordinate sets: Phase 1 the learning dynamics, Phase 2 the architecture. This is the search-space narrowing/handoff that makes the two-phase scheme work.

> **What you should see:** the Phase-2 sampled keys are exactly the `tunable_arch_params()` keys (with `features` appearing as `features__idx`), **disjoint** from the Phase-1 keys. Every Phase-2 trial's model config still carries the frozen Phase-1 learning rates / weight decays unchanged. The Phase-2 running-best objective is non-increasing.


In [ ]:
phase2_arch_space = model_config_class.tunable_arch_params()


class Phase2ApplyProbe:
    def __init__(self, model_config_cls, best_phase1_params):
        self.model_config_cls   = model_config_cls
        self.best_phase1_params = best_phase1_params
        self.sampler            = ParamSampler()

    def apply(self, trial):
        model_config = self.model_config_cls()
        arch_sampled = self.sampler.sample(trial, self.model_config_cls.tunable_arch_params())
        for parameter_name, parameter_value in self.best_phase1_params.items():
            if hasattr(model_config, parameter_name):
                setattr(model_config, parameter_name, parameter_value)
        for parameter_name, parameter_value in arch_sampled.items():
            setattr(model_config, parameter_name, parameter_value)
        return model_config, arch_sampled


phase2_probe = Phase2ApplyProbe(model_config_class, decoded_phase1_params)


def phase2_surrogate_objective(trial):
    model_config, arch_sampled = phase2_probe.apply(trial)
    bottleneck_penalty = {1: 0.05, 2: 0.0, 4: 0.04}[arch_sampled["bottleneck_factor"]]
    activation_penalty = {"relu": 0.02, "leaky_relu": 0.01, "gelu": 0.0, "silu": 0.005}[arch_sampled["activation"]]
    normalization_penalty = {"batch": 0.0, "instance": 0.03, "group": 0.02}[arch_sampled["normalization"]]
    feature_penalty = 0.01 * (len(arch_sampled["features"]) - 4) ** 2
    trial.set_user_attr("encoder_lr_seen", model_config.encoder_lr)
    return float(0.10 + bottleneck_penalty + activation_penalty + normalization_penalty + feature_penalty)


phase2_loop_trial_budget = 60
phase2_loop_study = optuna.create_study(
    direction = "minimize",
    sampler   = optuna.samplers.TPESampler(seed=42, multivariate=True, constant_liar=True),
    pruner    = optuna.pruners.MedianPruner(
        n_startup_trials = tuning_configuration.phase2.pruner_n_startup_trials,
        n_warmup_steps   = tuning_configuration.phase2.pruner_n_warmup_steps,
    ),
)
phase2_loop_study.optimize(phase2_surrogate_objective, n_trials=phase2_loop_trial_budget)


In [ ]:
phase2_completed_trials = [t for t in phase2_loop_study.trials if t.state == optuna.trial.TrialState.COMPLETE]
phase2_objective_values = np.array([t.value for t in phase2_completed_trials])

phase1_space_keys = set(model_config_class.tunable_lr_params().keys())
phase2_space_keys = set(model_config_class.tunable_arch_params().keys())
overlapping_keys  = phase1_space_keys & phase2_space_keys

representative_phase2_trial = phase2_loop_study.best_trial
encoder_lr_seen_in_phase2   = representative_phase2_trial.user_attrs["encoder_lr_seen"]

print("Phase-1 search-space keys  :", sorted(phase1_space_keys))
print("Phase-2 search-space keys  :", sorted(phase2_space_keys))
print("overlapping keys (must be set()):", overlapping_keys)
print()
print("Phase-2 completed trials   :", len(phase2_completed_trials))
print("Phase-2 best value         :", phase2_loop_study.best_value)
print("Phase-2 best params        :", representative_phase2_trial.params)
print()
print("frozen encoder_lr from Phase 1 :", decoded_phase1_params["encoder_lr"])
print("encoder_lr seen in Phase-2 cfg :", encoder_lr_seen_in_phase2)


In [ ]:
figure, (axis_history, axis_box) = plt.subplots(1, 2, figsize=(10.5, 3.8))

phase2_trial_indices = np.array([t.number for t in phase2_completed_trials])
phase2_running_best  = np.minimum.accumulate(phase2_objective_values)
axis_history.scatter(phase2_trial_indices, phase2_objective_values, color=palette_neutral, s=18, alpha=0.6, label="trial objective")
axis_history.plot(phase2_trial_indices, phase2_running_best, color=palette_phase2, linewidth=1.8, label="running best")
axis_history.set_xlabel("trial index")
axis_history.set_ylabel("best validation loss (surrogate)")
axis_history.set_title("Stage 7 — Phase 2: objective vs trial")
axis_history.legend(frameon=False)

activation_choices = phase2_arch_space["activation"]["choices"]
grouped_losses = []
for activation_name in activation_choices:
    grouped_losses.append([t.value for t in phase2_completed_trials if t.params["activation"] == activation_name])
axis_box.boxplot(grouped_losses, labels=activation_choices, patch_artist=True, boxprops=dict(facecolor=palette_phase2, alpha=0.6))
axis_box.set_xlabel("activation (categorical)")
axis_box.set_ylabel("validation loss (surrogate)")
axis_box.set_title("Stage 7 — Phase 2: objective by activation choice")

figure.tight_layout()
figure.savefig(figure_directory / "stage_7_phase2_loop.pdf")
plt.show()


In [ ]:
figure, (axis_space, axis_widths) = plt.subplots(1, 2, figsize=(10.5, 3.6))

phase1_count = len(model_config_class.tunable_lr_params())
phase2_count = len(model_config_class.tunable_arch_params())
axis_space.bar(["Phase 1\n(learning)", "Phase 2\n(architecture)"], [phase1_count, phase2_count], color=[palette_phase1, palette_phase2], edgecolor="black", linewidth=0.6)
axis_space.set_ylabel("number of tunable dimensions")
axis_space.set_title("Stage 7 — Search-space narrowing across phases")

feature_choice_lengths = [len(choice) for choice in phase2_arch_space["features"]["choices"]]
axis_widths.bar(np.arange(len(feature_choice_lengths)), feature_choice_lengths, color=palette_neutral, edgecolor="black", linewidth=0.6)
axis_widths.set_xticks(np.arange(len(feature_choice_lengths)))
axis_widths.set_xticklabels([f"opt {i}" for i in range(len(feature_choice_lengths))])
axis_widths.set_xlabel("indexed_categorical feature option")
axis_widths.set_ylabel("encoder depth (#stages)")
axis_widths.set_title("Stage 7 — Phase-2 'features' choices")

figure.tight_layout()
figure.savefig(figure_directory / "stage_7_search_space_narrowing.pdf")
plt.show()


In [ ]:
phase1_best_preserved = all(
    abs(t.user_attrs["encoder_lr_seen"] - decoded_phase1_params["encoder_lr"]) < 1e-12
    for t in phase2_completed_trials
)
phase2_running_best_monotone = bool(np.all(np.diff(phase2_running_best) <= 1e-12))
phase2_keys_match_space      = set(representative_phase2_trial.params.keys()) <= (phase2_space_keys | {"features__idx"})

stage_7_checks = [
    ("Phase-1 / Phase-2 spaces disjoint",        len(overlapping_keys) == 0),
    ("Phase-2 keys are architectural only",      phase2_keys_match_space),
    ("Phase-1 best frozen in every Phase-2 trial", phase1_best_preserved),
    ("Phase-2 recorded == requested budget",     len(phase2_loop_study.trials) == phase2_loop_trial_budget),
    ("Phase-2 best <= max completed",            phase2_loop_study.best_value <= phase2_objective_values.max() + 1e-9),
    ("Phase-2 running best non-increasing",      phase2_running_best_monotone),
]
for stage_7_description, stage_7_condition in stage_7_checks:
    print(f"[{'PASS' if stage_7_condition else 'FAIL'}] {stage_7_description}")


### Common Mistakes — Stage 7: Phase-2 tuning loop

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Phase 2 re-tunes the learning rate | The two search spaces overlap, or Phase-1 best applied **before** arch sampling and then overwritten | Assert `tunable_lr_params().keys() ∩ tunable_arch_params().keys() == ∅` and inspect the order in `_apply_params` |
| Phase-1 best silently ignored | A Phase-1 key not present on the model config (`hasattr` guard skips it) — e.g. renamed field | Compare decoded keys against the model config attributes; the guard hides typos |
| Phase 2 explores the same arch every trial | `features` left as plain `categorical` so list choices fail and collapse to a default | Confirm `features` is `indexed_categorical` and appears as `features__idx` in trial params |
| Phase-2 best worse than Phase-1 baseline | Phase 2 changed architecture but kept Phase-1 LR tuned for a *different* architecture (interaction) | Compare Phase-2 best value against Phase-1 best; consider re-tuning LR jointly if large |
| Direction inconsistent between phases | One study created with a different `direction` | Assert both studies are `MINIMIZE` (Stage 1 + here) |

**Passing to end-to-end summary:** `phase2_loop_study` — the finished architecture study; its `best_trial` merged with the Phase-1 best yields the final winning configuration.


---
## End-to-End Summary — assembled two-phase tuning result

This final group cross-checks the orchestrated flow. The production scheduler (`main/tune.py`) runs Phase 1, extracts and decodes the best trial into `phase1_best.json`, then runs Phase 2 seeded with those params and writes a merged `best_config.json` with `{**best_p1_trial.params, **best_p2_trial.params}`. Below we assemble the same merged configuration from the stage objects built above and assert overall consistency.


In [ ]:
merged_best_configuration = {**best_phase1_trial.params, **representative_phase2_trial.params}

summary_rows = [
    ("Phase-1 trials run",         len(phase1_loop_study.trials)),
    ("Phase-1 completed",          len(phase1_completed_trials)),
    ("Phase-1 best trial",         best_phase1_trial.number),
    ("Phase-1 best value",         round(best_phase1_trial.value, 6)),
    ("Phase-1 tunable dims",       len(model_config_class.tunable_lr_params())),
    ("Phase-2 trials run",         len(phase2_loop_study.trials)),
    ("Phase-2 completed",          len(phase2_completed_trials)),
    ("Phase-2 best trial",         representative_phase2_trial.number),
    ("Phase-2 best value",         round(phase2_loop_study.best_value, 6)),
    ("Phase-2 tunable dims",       len(model_config_class.tunable_arch_params())),
    ("Merged config keys",         len(merged_best_configuration)),
    ("Search spaces disjoint",     len(set(model_config_class.tunable_lr_params()) & set(model_config_class.tunable_arch_params())) == 0),
]

label_width = max(len(label) for label, _ in summary_rows)
print("Two-Phase Tuning — End-to-End Summary")
print("=" * (label_width + 22))
for summary_label, summary_value in summary_rows:
    print(f"{summary_label:<{label_width}} : {summary_value}")
print()
print("merged best configuration (Phase-1 learning + Phase-2 architecture):")
for parameter_name, parameter_value in merged_best_configuration.items():
    print(f"  {parameter_name:<18}: {parameter_value}")


In [ ]:
figure, axis = plt.subplots(figsize=(7.5, 3.6))
phase_labels = ["Phase 1 best\n(learning dynamics)", "Phase 2 best\n(architecture)"]
phase_values = [best_phase1_trial.value, phase2_loop_study.best_value]
axis.bar(phase_labels, phase_values, color=[palette_phase1, palette_phase2], edgecolor="black", linewidth=0.6)
for bar_index, bar_value in enumerate(phase_values):
    axis.text(bar_index, bar_value, f"{bar_value:.4f}", ha="center", va="bottom")
axis.set_ylabel("best validation loss (surrogate)")
axis.set_title("End-to-end — best objective per tuning phase")

figure.tight_layout()
figure.savefig(figure_directory / "stage_8_end_to_end_summary.pdf")
plt.show()


In [ ]:
end_to_end_checks = [
    ("Phase-1 study has a best trial",            best_phase1_trial is not None),
    ("Phase-2 study has a best trial",            representative_phase2_trial is not None),
    ("Decoded Phase-1 params non-empty",          len(decoded_phase1_params) > 0),
    ("Merged config covers both phases",          len(merged_best_configuration) >= len(decoded_phase1_params)),
    ("Search spaces remain disjoint",             len(set(model_config_class.tunable_lr_params()) & set(model_config_class.tunable_arch_params())) == 0),
    ("Both studies minimise",                     phase1_loop_study.direction == optuna.study.StudyDirection.MINIMIZE and phase2_loop_study.direction == optuna.study.StudyDirection.MINIMIZE),
    ("TuningPipeline exposes both phase entry points", hasattr(TuningPipeline, "run_phase1") and hasattr(TuningPipeline, "run_phase2")),
]
for end_to_end_description, end_to_end_condition in end_to_end_checks:
    print(f"[{'PASS' if end_to_end_condition else 'FAIL'}] {end_to_end_description}")

print()
print("All stage figures written to:", figure_directory)
